# 05 — Shor [[9, 1, 3]] code

Phase 5 demo. We verify the Shor stabilizers form a valid code, show that every weight-1 Pauli error is corrected, observe a representative weight-2 logical failure, and plot Monte-Carlo logical error rate vs `p` on log-log axes (slope ~ 2).

Read alongside `notes/07-shor-9qubit.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qec.pauli import Pauli
from qec.codes import shor

plt.rcParams["figure.figsize"] = (6, 4)


## 1. The stabilizer generators

Six Z-type (handle X errors within blocks) and two X-type (handle Z errors between blocks).


In [ ]:
print("Z-type stabilizers:")
for s in shor.Z_STABILIZERS:
    print(" ", s)
print("X-type stabilizers:")
for s in shor.X_STABILIZERS:
    print(" ", s)
print("logical X:", shor.LOGICAL_X)
print("logical Z:", shor.LOGICAL_Z)


## 2. Validity check: pairwise commutation + logical anticommutation

In [ ]:
from itertools import combinations
ok = True
for a, b in combinations(shor.ALL_STABILIZERS, 2):
    if not a.commutes_with(b):
        print("FAIL:", a, b); ok = False
for s in shor.ALL_STABILIZERS:
    if not shor.LOGICAL_X.commutes_with(s) or not shor.LOGICAL_Z.commutes_with(s):
        print("logical anticommutes with", s); ok = False
if shor.LOGICAL_X.commutes_with(shor.LOGICAL_Z):
    print("logical X and Z commute (should anticommute)"); ok = False
print("valid stabilizer code:", ok)
assert ok


## 3. Every weight-1 Pauli is corrected

There are 9 qubits × 3 non-identity single-qubit Paulis = 27 weight-1 errors. All of them must produce non-zero syndromes that uniquely identify the recovery.


In [ ]:
import itertools
n_corrected = 0
for E in shor._enumerate_paulis(1):
    residual = shor.correct(E)
    lx, lz = shor.is_logical_failure(residual)
    if not lx and not lz:
        n_corrected += 1
print(f"weight-1 errors corrected: {n_corrected}/27")
assert n_corrected == 27
print("OK")


## 4. A weight-2 error that fools the decoder

X errors on two qubits in the *same* block can produce the same syndrome as a single X on the third qubit of that block — leading to a logical X within the block, then a phase-flip-code correction on the wrong block, etc.


In [ ]:
import numpy as np
# X errors on qubits 0 and 1 of block 0 -> decoder applies X on qubit 2 ->
# residual = X_0 X_1 X_2 = a stabilizer's logical neighbour. Let's see.
x = np.zeros(9, dtype=np.int8); z = np.zeros(9, dtype=np.int8)
x[0] = x[1] = 1
E = Pauli(x, z, 0)
res = shor.correct(E)
lx, lz = shor.is_logical_failure(res)
print(f"input error : {E}")
print(f"residual    : {res}")
print(f"logical X / logical Z applied : {lx} / {lz}")


## 5. Logical error rate, log-log slope ~ 2

In [ ]:
rng = np.random.default_rng(2026)
ps = np.array([0.005, 0.01, 0.02, 0.03, 0.05, 0.07])
shots = 30_000
pls = np.array([shor.monte_carlo_logical_error(p, shots, rng=rng) for p in ps])
print("p     P_L")
for p, pl in zip(ps, pls):
    print(f"{p:.3f}  {pl:.4f}")

# Fit slope on log-log
mask = pls > 0
slope = np.polyfit(np.log(ps[mask]), np.log(pls[mask]), 1)[0]
print(f"log-log slope: {slope:.2f}  (theory ~ 2)")

plt.loglog(ps, pls, "C0o-", label="Shor [[9,1,3]] (MC)")
plt.loglog(ps, ps, "k:", label="no encoding")
ref = pls[mask][0] * (ps / ps[mask][0])**2
plt.loglog(ps, ref, "C0--", alpha=0.5, label="slope-2 reference")
plt.xlabel("p (per-qubit Pauli error rate)")
plt.ylabel("P_L(p)")
plt.title("Shor code: logical error rate vs physical")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()


## What this notebook gates

- The 8 stabilizers form a valid 1-logical-qubit code.
- Every weight-1 Pauli is corrected exactly.
- A representative weight-2 input causes a logical failure (distance 3).
- Logical error rate scales as ~ p^2 below threshold.

Next: `06_steane_code.ipynb`.
